In [356]:
import mlflow
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split

In [357]:
import mlflow

while mlflow.active_run():
    mlflow.end_run()

In [358]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("FNN Classification")

params ={"batch":5,
         "learning_rate":0.001,
         "epochs":40,
         "random_seed":42,
         "test_size":0.2
         }

with mlflow.start_run(run_name="small_ml") as run:
    mlflow.log_params(params)

🏃 View run small_ml at: http://localhost:5001/#/experiments/5/runs/94c9bad4c0174237ba9c9b2e4038fbea
🧪 View experiment at: http://localhost:5001/#/experiments/5


In [359]:
import sys
sys.path.append("..")

from load_imbd import load_imdb_split

train_fn = load_imdb_split("/Users/benjaminbrooke/PycharmProjects/MLOps/LLMOps/aclImdb/train")

In [360]:
print(next(iter(train_fn)))

{'text': 'For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe Piscopo is actually funny! Maureen Stapleton is a scene stealer. The Moroni character is an absolute scream. Watch for Alan "The Skipper" Hale jr. as a police Sgt.', 'label': 1}


In [361]:
print(train_fn)

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


In [362]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = data.target

In [363]:
type(X)

numpy.ndarray

In [364]:
print(set(y))

{np.int64(0), np.int64(1)}


In [365]:
print(next(iter(X)))
print(next(iter(y)))

print(len(X))

print(len(next(iter(X))))

print(X.shape,y.shape)



[1.799e+01 1.038e+01 1.228e+02 1.001e+03 1.184e-01 2.776e-01 3.001e-01
 1.471e-01 2.419e-01 7.871e-02 1.095e+00 9.053e-01 8.589e+00 1.534e+02
 6.399e-03 4.904e-02 5.373e-02 1.587e-02 3.003e-02 6.193e-03 2.538e+01
 1.733e+01 1.846e+02 2.019e+03 1.622e-01 6.656e-01 7.119e-01 2.654e-01
 4.601e-01 1.189e-01]
0
569
30
(569, 30) (569,)


In [366]:
class MyClassificationFNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.lin1 = nn.Linear(30, 120)
        self.relu1 = nn.ReLU()
        self.norm1 = nn.BatchNorm1d(120)

        self.lin2 = nn.Linear(120, 240)
        self.relu2 = nn.ReLU()
        self.norm2 = nn.BatchNorm1d(240)

        self.lin3 = nn.Linear(240, 240)
        self.relu3 = nn.ReLU()
        self.norm3 = nn.BatchNorm1d(240)

        self.lin4 = nn.Linear(240, 120)
        self.relu4 = nn.ReLU()
        self.norm4 = nn.BatchNorm1d(120)

        self.lin5 = nn.Linear(120, 2)


    def forward(self, x):
        x = self.norm1(self.relu1(self.lin1(x)))
        x = self.norm2(self.relu2(self.lin2(x)))
        x = self.norm3(self.relu3(self.lin3(x)))
        x = self.norm4(self.relu4(self.lin4(x)))

        return self.lin5(x)

In [367]:
class_model = MyClassificationFNN()

In [368]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(class_model.parameters(), lr=params["learning_rate"])

In [369]:
import sklearn
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=params["test_size"],
                                                    random_state=params["random_seed"])

In [370]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

In [371]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

In [372]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=params["batch"], shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=params["batch"], shuffle=False)

In [373]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, dataloader):
    model.eval()

    predictions, true_labels = [], []

    with torch.no_grad():

        for X_batch_eval, y_batch_eval in dataloader:

            output_eval = class_model(X_batch_eval)

            loss_eval = loss_fn(output_eval, y_batch_eval)

            preds = torch.argmax(output_eval, dim=1)

            predictions.extend(preds.tolist())
            true_labels.extend(y_batch_eval.tolist())

    accuracy_score_ = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average="macro")

    return accuracy_score_, precision, recall, f1

In [374]:
epochs = 10
from tqdm import tqdm

for epoch in range(epochs):

    running_loss = 0.0

    for i, (X_batch,y_batch) in enumerate(train_loader):


        optimizer.zero_grad()

        output = class_model(X_batch)

        loss = loss_fn(output, y_batch)

        loss.backward()

        optimizer.step()

        if i % 5 == 0 and i > 0:

                avg_loss = running_loss / 5

                mlflow.log_metric("train_loss", avg_loss, step=epoch)

                running_loss = 0.0

        accuracy, precision, recall, f1 = evaluate_model(class_model, test_loader)

        mlflow.log_metrics({"accuracy":accuracy,"precision":precision,"recall":recall,"f1":f1})

torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])


/opt/anaconda3/envs/mlops/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/mlops/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/mlops/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Size([5])
torch.Size([5, 30]) torch.Si